In [ ]:
"""
FloodXplain - complete flood-risk model in ONE file
====================================================
Drop this file into your project.

    # 1) train once (Colab GPU recommended, ~20-40 min)
    python floodxplain_model.py --train

    # 2) predict for any place in India (or your current GPS position)
    python floodxplain_model.py --predict "Saravanampatti"
    python floodxplain_model.py --lat 11.07 --lon 77.00

    # 3) use inside your app (Flask / FastAPI / Streamlit ...)
    from floodxplain_model import FloodPredictor
    fp = FloodPredictor()                       # loads the trained files
    result = fp.predict(place="Coimbatore")     # or lat=..., lon=...
    strip  = fp.hourly_strip(place="Coimbatore")  # 24-hour risk strip + reasons

What the file does
------------------
 1. DATA        downloads real hourly ERA5 weather (Open-Meteo) for ~20 flood-prone
                and dry Indian locations (2019-today). If the download is not possible
                it falls back to SYNTHETIC monsoon data (clearly flagged).
 2. FEATURES    rainfall accumulations (3h ... 7 days), soil wetness, pressure trend,
                humidity, time-of-year, terrain (elevation, slope, relief).
 3. LABELS      verified labels if you supply them (VERIFIED_LABEL_CSV), otherwise a
                PROXY label from IMD rainfall rules (see the honesty note below).
 4. MODELS      ANN + LSTM + GRU, each trained on the past 24 h -> "flood-risk event
                in the next 24 h".
 5. COMBINING   three techniques compared on held-out validation data: simple mean,
                PR-AUC-weighted mean, and stacking (logistic-regression meta-model).
                The best one becomes the single final model.
 6. TESTING     chronological test set + comparison table (accuracy, precision, recall,
                F1, ROC-AUC, PR-AUC, Brier error, MAE) and a "persistence" baseline.
 7. PREDICTION  place name or GPS -> live weather -> probability + risk level.
 8. EXPLAIN     Integrated Gradients on the final ensemble -> top drivers + a
                plain-language sentence.

HONESTY NOTE
------------
No public dataset gives hour-by-hour verified flood labels for every Indian location.
Unless you provide VERIFIED_LABEL_CSV, the label is a rainfall-rule PROXY, so the
reported accuracy shows how well the models learn/forecast that rule, not how well
they match real inundation. The comparison table includes a persistence baseline and
a "predict-no-flood-always" accuracy so you can see whether a high accuracy number
means anything. Treat the output as an early-warning RISK SCORE, not an official
flood forecast; always follow IMD / CWC / local authority alerts.
"""

'\nFloodXplain - complete flood-risk model in ONE file\n====================================================\nDrop this file into your project.\n\n    # 1) train once (Colab GPU recommended, ~20-40 min)\n    python floodxplain_model.py --train\n\n    # 2) predict for any place in India (or your current GPS position)\n    python floodxplain_model.py --predict "Saravanampatti"\n    python floodxplain_model.py --lat 11.07 --lon 77.00\n\n    # 3) use inside your app (Flask / FastAPI / Streamlit ...)\n    from floodxplain_model import FloodPredictor\n    fp = FloodPredictor()                       # loads the trained files\n    result = fp.predict(place="Coimbatore")     # or lat=..., lon=...\n    strip  = fp.hourly_strip(place="Coimbatore")  # 24-hour risk strip + reasons\n\nWhat the file does\n------------------\n 1. DATA        downloads real hourly ERA5 weather (Open-Meteo) for ~20 flood-prone\n                and dry Indian locations (2019-today). If the download is not possible\n     

In [ ]:

from __future__ import annotations

import argparse
import json
import math
import os
import sys
import time
import warnings
import zlib
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras import callbacks, layers, models
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score,
                             brier_score_loss, confusion_matrix, f1_score,
                             mean_absolute_error, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler

# ============================ CONFIGURATION ===============================
SEED = 42
HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path(".")
ARTIFACT_DIR = HERE / "floodxplain_artifacts"      # trained models are saved here
CACHE_DIR = HERE / "floodxplain_data_cache"        # downloaded weather is cached here

WINDOW = 24            # hours of history given to the models
HORIZON = 24           # predict: risk event within the NEXT 24 hours
START_YEAR = 2019
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15                  # remaining 15 % = test

# Proxy-label rules (IMD classes: heavy rain >= 64.5 mm / 24 h)
HEAVY_24H_MM = 64.5
BURST_3H_MM = 30.0
SATURATED_SOIL_PERCENTILE = 90

VERIFIED_LABEL_CSV = None   # optional CSV: location,time,flood_label  (1 = flood)

EPOCHS = 20
BATCH = 512
PATIENCE = 4
NEG_STRIDE = 3          # keep every 3rd non-event training window (all events kept)
DATA_MODE = "auto"      # "auto" | "real" | "synthetic"

# name, latitude, longitude, wetness (only used by the synthetic fallback)
LOCATIONS = [
    ("Coimbatore", 11.02, 76.96, 0.6), ("Saravanampatti", 11.07, 77.00, 0.6),
    ("Chennai", 13.08, 80.27, 0.9), ("Alappuzha", 9.49, 76.33, 1.2),
    ("Wayanad", 11.69, 76.08, 1.3), ("Bengaluru", 12.97, 77.59, 0.6),
    ("Hyderabad", 17.38, 78.48, 0.6), ("Mumbai", 19.07, 72.88, 1.4),
    ("Surat", 21.17, 72.83, 0.9), ("Kolkata", 22.57, 88.36, 1.0),
    ("Cuttack", 20.46, 85.88, 1.0), ("Guwahati", 26.14, 91.74, 1.4),
    ("Dibrugarh", 27.48, 94.91, 1.4), ("Patna", 25.59, 85.14, 0.8),
    ("Darbhanga", 26.15, 85.90, 0.8), ("Delhi", 28.61, 77.21, 0.5),
    ("Dehradun", 30.32, 78.03, 1.0), ("Srinagar", 34.08, 74.80, 0.6),
    ("Jaipur", 26.91, 75.79, 0.3), ("Jaisalmer", 26.91, 70.92, 0.1),
]

HOURLY_VARS = ["temperature_2m", "relative_humidity_2m", "precipitation",
               "surface_pressure", "wind_speed_10m", "soil_moisture_0_to_7cm"]

DYNAMIC_FEATURES = (
    HOURLY_VARS
    + [f"rain_sum_{k}h" for k in (3, 6, 12, 24, 48, 72, 168)]
    + ["rain_max_6h", "rain_hours_24h", "soil_mean_24h", "soil_change_24h",
       "pressure_change_3h", "pressure_change_24h", "humidity_mean_6h",
       "temp_mean_24h", "hour_sin", "hour_cos", "doy_sin", "doy_cos"])
STATIC_FEATURES = ["elevation_m", "slope_degrees", "relief_m", "elevation_above_min_m"]
FEATURES = DYNAMIC_FEATURES + STATIC_FEATURES

tf.keras.utils.set_random_seed(SEED)


# ============================== HTTP HELPERS ==============================
def _get_json(url, params, tries=5, timeout=90):
    last = None
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code == 429:                      # rate limited
                time.sleep(30 * attempt)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as exc:                          # noqa: BLE001
            last = exc
            time.sleep(3 * attempt)
    raise RuntimeError(f"Request failed: {url} ({last})")


def geocode(place: str):
    """Place name -> (lat, lon, pretty label). India only."""
    data = _get_json("https://geocoding-api.open-meteo.com/v1/search",
                     {"name": place, "count": 1, "countryCode": "IN"})
    if not data.get("results"):
        raise ValueError(f"'{place}' was not found in India. Check the spelling.")
    r = data["results"][0]
    return r["latitude"], r["longitude"], f'{r["name"]}, {r.get("admin1", "")}'.strip(", ")


def fetch_static(lat: float, lon: float, fallback_seed: str | None = None) -> dict:
    """Terrain features from a 7x7 elevation grid (90 m spacing), as in Phase 5C."""
    G, S = 7, 90.0
    dlat = S / 111320.0
    dlon = S / (111320.0 * math.cos(math.radians(lat)))
    la = [lat + (r - G // 2) * dlat for r in range(G) for _ in range(G)]
    lo = [lon + (c - G // 2) * dlon for _ in range(G) for c in range(G)]
    try:
        data = _get_json("https://api.open-meteo.com/v1/elevation",
                         {"latitude": ",".join(f"{v:.6f}" for v in la),
                          "longitude": ",".join(f"{v:.6f}" for v in lo)})
        grid = np.array(data["elevation"], dtype=float).reshape(G, G)
    except Exception:                                      # offline / synthetic mode
        rng = np.random.default_rng(zlib.crc32((fallback_seed or f"{lat}{lon}").encode()))
        base = rng.uniform(5, 900)
        grid = base + rng.normal(0, 3, (G, G)).cumsum(0).cumsum(1) * 0.1
    gy, gx = np.gradient(grid, S, S)
    c = G // 2
    slope = float(np.degrees(np.arctan(np.hypot(gx[c, c], gy[c, c]))))
    return {"elevation_m": float(grid[c, c]), "slope_degrees": slope,
            "relief_m": float(grid.max() - grid.min()),
            "elevation_above_min_m": float(grid[c, c] - grid.min())}


def fetch_weather(lat: float, lon: float, on_date: str | None = None,
                  history_days: int = 10) -> pd.DataFrame:
    """Recent hourly weather. Live: forecast API (past + next days). Date: ERA5 archive."""
    params = {"latitude": lat, "longitude": lon, "hourly": ",".join(HOURLY_VARS),
              "timezone": "Asia/Kolkata"}
    if on_date is None:
        params.update(past_days=history_days, forecast_days=3)
        url = "https://api.open-meteo.com/v1/forecast"
    else:
        end = pd.Timestamp(on_date)
        params.update(start_date=(end - pd.Timedelta(days=history_days)).strftime("%Y-%m-%d"),
                      end_date=end.strftime("%Y-%m-%d"), models="era5")
        url = "https://archive-api.open-meteo.com/v1/archive"
    w = pd.DataFrame(_get_json(url, params)["hourly"])
    w["time"] = pd.to_datetime(w["time"])
    return w.sort_values("time").reset_index(drop=True)


# ============================ DATA ACQUISITION ============================
def _download_location(name, lat, lon, start, end) -> pd.DataFrame:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    parts = []
    for year in range(start.year, end.year + 1):
        s = max(start, pd.Timestamp(year, 1, 1))
        e = min(end, pd.Timestamp(year, 12, 31))
        f = CACHE_DIR / f"{name}_{year}.csv"
        if f.exists():
            parts.append(pd.read_csv(f, parse_dates=["time"]))
            continue
        data = _get_json("https://archive-api.open-meteo.com/v1/archive",
                         {"latitude": lat, "longitude": lon,
                          "start_date": s.strftime("%Y-%m-%d"),
                          "end_date": e.strftime("%Y-%m-%d"),
                          "hourly": ",".join(HOURLY_VARS),
                          "timezone": "Asia/Kolkata", "models": "era5"}, timeout=180)
        df = pd.DataFrame(data["hourly"])
        df["time"] = pd.to_datetime(df["time"])
        df.to_csv(f, index=False)
        parts.append(df)
        time.sleep(1.0)
    return pd.concat(parts, ignore_index=True)


def _synthetic_location(name, lat, lon, wetness, start, end) -> pd.DataFrame:
    """Monsoon-like synthetic hourly weather (fallback only)."""
    rng = np.random.default_rng(zlib.crc32(name.encode()))
    t = pd.date_range(start, end + pd.Timedelta(hours=23), freq="h")
    n = len(t)
    doy = t.dayofyear.values
    season = 0.12 + 0.88 * np.exp(-((doy - 200) / 55.0) ** 2)
    rain = np.zeros(n)
    for day in range(n // 24):
        if rng.random() < min(0.9, 0.55 * season[day * 24] * wetness):
            dur = int(rng.integers(2, 14))
            s0 = int(rng.integers(0, 24))
            scale = 3.0 * season[day * 24] * wetness * (4.0 if rng.random() < 0.06 else 1.0)
            for h in range(dur):
                i = day * 24 + s0 + h
                if i < n:
                    rain[i] += rng.gamma(1.4, scale)
    soil = np.zeros(n)
    s = 0.2
    for i in range(n):
        s = float(np.clip(0.985 * s + 0.010 * rain[i] + 0.0006, 0.08, 0.5))
        soil[i] = s
    hour = t.hour.values
    wet = (pd.Series(rain).rolling(6, min_periods=1).mean().values > 0.1).astype(float)
    return pd.DataFrame({
        "time": t,
        "temperature_2m": 27 + 5 * np.sin((hour - 9) / 24 * 2 * np.pi) - 3 * wet + rng.normal(0, .5, n),
        "relative_humidity_2m": np.clip(60 + 30 * wet + rng.normal(0, 5, n), 20, 100),
        "precipitation": rain,
        "surface_pressure": 960 - 3 * wet - 0.05 * pd.Series(rain).rolling(6, min_periods=1).sum().values
                            + rng.normal(0, .6, n),
        "wind_speed_10m": np.clip(8 + 4 * wet + rng.normal(0, 2, n), 0, None),
        "soil_moisture_0_to_7cm": soil})


def load_training_weather(mode=DATA_MODE):
    """Returns ({name: (weather_df, static_dict)}, source_label)."""
    end = (pd.Timestamp.today().normalize() - pd.Timedelta(days=7))
    start = pd.Timestamp(START_YEAR, 1, 1)
    out, source = {}, "REAL (Open-Meteo ERA5)"
    if mode in ("auto", "real"):
        for name, lat, lon, _ in LOCATIONS:
            try:
                print(f"  downloading {name} ...")
                out[name] = (_download_location(name, lat, lon, start, end),
                             fetch_static(lat, lon, name))
            except Exception as exc:                       # noqa: BLE001
                print(f"  ! {name} failed: {exc}")
        if len(out) < 8:
            if mode == "real":
                raise RuntimeError("Too few locations downloaded. Check internet access.")
            print("Real download incomplete -> using SYNTHETIC data.")
            out = {}
    if not out:
        source = "SYNTHETIC (fallback) - accuracy is NOT meaningful for real floods"
        for name, lat, lon, wet in LOCATIONS:
            out[name] = (_synthetic_location(name, lat, lon, wet, start, end),
                         fetch_static(lat, lon, name))
    return out, source


# ========================= FEATURE ENGINEERING ============================
def engineer(weather: pd.DataFrame, static: dict) -> pd.DataFrame:
    """Hourly weather (+ static terrain) -> model features. Uses data up to and
    INCLUDING each hour only (no future information)."""
    d = weather.sort_values("time").reset_index(drop=True).copy()
    d[HOURLY_VARS] = d[HOURLY_VARS].ffill().bfill()
    p = d["precipitation"].clip(lower=0)
    for k in (3, 6, 12, 24, 48, 72, 168):
        d[f"rain_sum_{k}h"] = p.rolling(k, min_periods=k).sum()
    d["rain_max_6h"] = p.rolling(6, min_periods=6).max()
    d["rain_hours_24h"] = (p > 0.1).astype(float).rolling(24, min_periods=24).sum()
    s = d["soil_moisture_0_to_7cm"]
    d["soil_mean_24h"] = s.rolling(24, min_periods=24).mean()
    d["soil_change_24h"] = s - s.shift(24)
    pr = d["surface_pressure"]
    d["pressure_change_3h"] = pr - pr.shift(3)
    d["pressure_change_24h"] = pr - pr.shift(24)
    d["humidity_mean_6h"] = d["relative_humidity_2m"].rolling(6, min_periods=6).mean()
    d["temp_mean_24h"] = d["temperature_2m"].rolling(24, min_periods=24).mean()
    d["hour_sin"] = np.sin(2 * np.pi * d["time"].dt.hour / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d["time"].dt.hour / 24)
    d["doy_sin"] = np.sin(2 * np.pi * d["time"].dt.dayofyear / 365.25)
    d["doy_cos"] = np.cos(2 * np.pi * d["time"].dt.dayofyear / 365.25)
    for k in STATIC_FEATURES:
        d[k] = static[k]
    return d.dropna(subset=FEATURES).reset_index(drop=True)


# ===================== DATASET (labels, split, windows) ===================
def prepare_dataset(per_location, source):
    frames = []
    for name, (w, st) in per_location.items():
        f = engineer(w, st)
        f["location"] = name
        frames.append(f)
    df = pd.concat(frames, ignore_index=True)

    times = np.sort(df["time"].unique())
    t_train = times[int(len(times) * TRAIN_FRAC)]
    t_val = times[int(len(times) * (TRAIN_FRAC + VAL_FRAC))]

    soil_thr = float(np.percentile(df.loc[df["time"] <= t_train, "soil_moisture_0_to_7cm"],
                                   SATURATED_SOIL_PERCENTILE))
    event = ((df["rain_sum_24h"] >= HEAVY_24H_MM) |
             ((df["rain_sum_3h"] >= BURST_3H_MM) &
              (df["soil_moisture_0_to_7cm"] >= soil_thr))).astype(int).values
    label_type = "PROXY (IMD rainfall rule)"
    if VERIFIED_LABEL_CSV:
        lab = pd.read_csv(VERIFIED_LABEL_CSV, parse_dates=["time"])
        m = df[["location", "time"]].merge(lab[["location", "time", "flood_label"]],
                                           on=["location", "time"], how="left")
        event = m["flood_label"].fillna(0).astype(int).values
        label_type = "VERIFIED"

    scaler = StandardScaler().fit(df.loc[df["time"] <= t_train, FEATURES])
    Xall = scaler.transform(df[FEATURES]).astype("float32")
    raw24 = df["rain_sum_24h"].values.astype("float32")
    tvals = df["time"].values

    ends, ys, t_end, t_lab, loc_ids = [], [], [], [], []
    for li, (name, g) in enumerate(df.groupby("location", sort=False)):
        idx = g.index.values                       # contiguous block for this location
        n = len(idx)
        if n < WINDOW + HORIZON + 5:
            continue
        ev = event[idx]
        c = np.concatenate([[0], np.cumsum(ev)])
        i = np.arange(WINDOW - 1, n - 1 - HORIZON)
        y = ((c[i + HORIZON + 1] - c[i + 1]) > 0).astype("float32")   # any event in t+1..t+H
        ends.append(idx[i]); ys.append(y)
        t_end.append(tvals[idx[i]]); t_lab.append(tvals[idx[i] + HORIZON])
        loc_ids.append(np.full(len(i), li))
    ends = np.concatenate(ends); y = np.concatenate(ys)
    t_end = np.concatenate(t_end); t_lab = np.concatenate(t_lab)

    tr = t_lab <= t_train                                   # purge: label horizon stays inside split
    va = (t_end > t_train) & (t_lab <= t_val)
    te = t_end > t_val
    print(f"\nData source : {source}\nLabel type  : {label_type}")
    print(f"Locations   : {df['location'].nunique()}   rows: {len(df):,}   features: {len(FEATURES)}")
    for nm, m in [("train", tr), ("val", va), ("test", te)]:
        print(f"  {nm:5s}: {m.sum():>9,} windows, flood-risk windows = "
              f"{int(y[m].sum()):>7,} ({y[m].mean()*100:.2f}%)")
    return {"df": df, "Xall": Xall, "raw24": raw24, "scaler": scaler,
            "ends": ends, "y": y, "t_end": t_end,
            "tr": tr, "va": va, "te": te,
            "source": source, "label_type": label_type, "soil_thr": soil_thr}


def make_ds(Xt, ends, y=None, sw=None, shuffle=False, batch=BATCH):
    offs = tf.constant(np.arange(-WINDOW + 1, 1), dtype=tf.int64)
    e = ends.astype(np.int64)
    if y is None:
        ds = tf.data.Dataset.from_tensor_slices(e).batch(batch)
        ds = ds.map(lambda b: tf.gather(Xt, b[:, None] + offs[None, :]))
    else:
        ds = tf.data.Dataset.from_tensor_slices((e, y.astype("float32"), sw.astype("float32")))
        if shuffle:
            ds = ds.shuffle(min(len(e), 200_000), seed=SEED)
        ds = ds.batch(batch)
        ds = ds.map(lambda b, yy, ww: (tf.gather(Xt, b[:, None] + offs[None, :]), yy, ww))
    return ds.prefetch(tf.data.AUTOTUNE)


# ================================ MODELS ==================================
def build_model(kind: str):
    inp = layers.Input((WINDOW, len(FEATURES)))
    if kind == "ANN":
        x = layers.Flatten()(inp)
        x = layers.Dense(128, activation="relu")(x); x = layers.Dropout(0.3)(x)
        x = layers.Dense(64, activation="relu")(x); x = layers.Dropout(0.2)(x)
    elif kind == "LSTM":
        x = layers.LSTM(64, return_sequences=True)(inp); x = layers.Dropout(0.3)(x)
        x = layers.LSTM(32)(x); x = layers.Dense(32, activation="relu")(x)
    elif kind == "GRU":
        x = layers.GRU(64, return_sequences=True)(inp); x = layers.Dropout(0.3)(x)
        x = layers.GRU(32)(x); x = layers.Dense(32, activation="relu")(x)
    else:
        raise ValueError(kind)
    out = layers.Dense(1, activation="sigmoid")(x)
    m = models.Model(inp, out, name=kind)
    m.compile(tf.keras.optimizers.Adam(5e-4), loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(curve="PR", name="pr_auc")])
    return m


# ============================ ENSEMBLE LOGIC ==============================
def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def fit_ensembles(P_a, y_a):
    """P_a: (n, 3) validation-A probabilities of ANN, LSTM, GRU."""
    k = P_a.shape[1]
    specs = {"mean": {"type": "weighted", "w": [1.0 / k] * k}}
    pr = np.array([max(average_precision_score(y_a, P_a[:, i]), 1e-3) for i in range(k)])
    specs["weighted"] = {"type": "weighted", "w": (pr / pr.sum()).tolist()}
    if len(set(y_a)) > 1:
        lr = LogisticRegression(C=1.0, max_iter=2000).fit(_logit(P_a), y_a)
        specs["stacking"] = {"type": "stack", "coef": lr.coef_[0].tolist(),
                             "intercept": float(lr.intercept_[0])}
    return specs


def apply_ensemble(spec, P):
    if spec["type"] == "weighted":
        return P @ np.array(spec["w"])
    z = _logit(P) @ np.array(spec["coef"]) + spec["intercept"]
    return 1.0 / (1.0 + np.exp(-z))


def best_threshold(y, p, min_recall=0.85):
    if len(set(y)) < 2:
        return 0.5
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-9)
    ok = rec[:-1] >= min_recall
    pick = np.where(ok, f1, -1)
    i = int(np.argmax(pick)) if ok.any() else int(np.argmax(f1))
    return float(thr[i])


def metrics_row(name, y, p, t):
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"model": name, "threshold": round(t, 3),
            "accuracy": accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0),
            "roc_auc": roc_auc_score(y, p) if len(set(y)) > 1 else np.nan,
            "pr_auc": average_precision_score(y, p) if y.sum() else np.nan,
            "brier_error": brier_score_loss(y, np.clip(p, 0, 1)),
            "MAE": mean_absolute_error(y, np.clip(p, 0, 1)),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn}


# =============================== TRAINING =================================
def train(mode=DATA_MODE, epochs=EPOCHS):
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    print("== 1/5 Loading data ==")
    per_loc, source = load_training_weather(mode)
    print("== 2/5 Feature engineering, labels, split ==")
    D = prepare_dataset(per_loc, source)
    Xt = tf.constant(D["Xall"])
    ends, y, tr, va, te = D["ends"], D["y"], D["tr"], D["va"], D["te"]

    # training windows: keep every event window, thin out the non-event ones
    tr_idx = np.where(tr)[0]
    pos = tr_idx[y[tr_idx] == 1]
    neg = tr_idx[y[tr_idx] == 0][::NEG_STRIDE]
    tr_idx = np.sort(np.concatenate([pos, neg]))
    w_pos = float(min(max(len(neg) / max(len(pos), 1), 1.0), 20.0))
    sw = np.where(y[tr_idx] == 1, w_pos, 1.0)
    ds_tr = make_ds(Xt, ends[tr_idx], y[tr_idx], sw, shuffle=True)
    va_idx, te_idx = np.where(va)[0], np.where(te)[0]
    ds_va = make_ds(Xt, ends[va_idx], y[va_idx], np.ones(len(va_idx)))
    ds_va_x = make_ds(Xt, ends[va_idx])
    ds_te_x = make_ds(Xt, ends[te_idx])

    print("== 3/5 Training ANN, LSTM, GRU ==")
    names = ["ANN", "LSTM", "GRU"]
    P_val, P_te = [], []
    for nm in names:
        print(f"\n--- {nm} ---")
        m = build_model(nm)
        m.fit(ds_tr, validation_data=ds_va, epochs=epochs, verbose=2,
              callbacks=[callbacks.EarlyStopping(monitor="val_pr_auc", mode="max",
                                                 patience=PATIENCE, restore_best_weights=True),
                         callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)])
        m.save(str(ARTIFACT_DIR / f"{nm}.keras"))
        P_val.append(m.predict(ds_va_x, verbose=0).ravel())
        P_te.append(m.predict(ds_te_x, verbose=0).ravel())
    P_val, P_te = np.stack(P_val, 1), np.stack(P_te, 1)
    y_va, y_te = y[va_idx], y[te_idx]

    print("== 4/5 Combining the three models ==")
    # validation split chronologically: A fits the combiner, B selects + sets threshold
    tv = D["t_end"][va_idx]
    cut = np.median(tv.astype("int64"))
    A, B = tv.astype("int64") <= cut, tv.astype("int64") > cut
    specs = fit_ensembles(P_val[A], y_va[A])
    rows_val = {}
    for nm, spec in specs.items():
        p = apply_ensemble(spec, P_val[B])
        rows_val[nm] = average_precision_score(y_va[B], p) if y_va[B].sum() else 0.0
    best_name = max(rows_val, key=rows_val.get)
    spec = specs[best_name]
    thr = best_threshold(y_va[B], apply_ensemble(spec, P_val[B]))
    print("Validation PR-AUC by combination method:",
          {k: round(v, 4) for k, v in rows_val.items()}, "-> chosen:", best_name)

    print("== 5/5 Testing and comparison ==")
    rows = []
    for i, nm in enumerate(names):
        t_i = best_threshold(y_va[B], P_val[B, i])
        rows.append(metrics_row(nm, y_te, P_te[:, i], t_i))
    for nm, sp in specs.items():
        t_i = best_threshold(y_va[B], apply_ensemble(sp, P_val[B]))
        rows.append(metrics_row(f"ensemble:{nm}" + ("  <== FINAL" if nm == best_name else ""),
                                y_te, apply_ensemble(sp, P_te), t_i))
    raw = D["raw24"][ends]
    s_va = np.clip(raw[va_idx] / HEAVY_24H_MM, 0, 1)
    s_te = np.clip(raw[te_idx] / HEAVY_24H_MM, 0, 1)
    rows.append(metrics_row("baseline: persistence (24h rain)", y_te, s_te,
                            best_threshold(y_va[B], s_va[B])))
    res = pd.DataFrame(rows)
    pd.set_option("display.width", 220)
    print("\n================= TEST COMPARISON =================")
    print(res.round(4).to_string(index=False))
    print(f"\nAlways-say-'no flood' accuracy on this test set: {1 - y_te.mean():.4f}")
    print("-> if model accuracy is close to that number, rely on recall / precision / PR-AUC.")
    res.round(5).to_csv(ARTIFACT_DIR / "test_comparison.csv", index=False)
    _plot(res)

    joblib.dump(D["scaler"], ARTIFACT_DIR / "scaler.joblib")
    cfg = {"features": FEATURES, "window": WINDOW, "horizon": HORIZON,
           "model_names": names, "ensemble_name": best_name, "ensemble": spec,
           "threshold": thr, "data_source": D["source"], "label_type": D["label_type"],
           "medians": {k: float(v) for k, v in pd.Series(D["scaler"].mean_, FEATURES).items()},
           "final_test": res[res["model"].str.contains("FINAL")].iloc[0].drop("model").to_dict()}
    json.dump(cfg, open(ARTIFACT_DIR / "config.json", "w"), indent=2, default=float)
    print(f"\nSaved final model to: {ARTIFACT_DIR}")
    return res


def _plot(res):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        r = res.set_index("model")[["recall", "precision", "f1", "pr_auc"]]
        r.plot(kind="bar", figsize=(11, 5), ylim=(0, 1))
        plt.title("Test comparison (higher is better)"); plt.xticks(rotation=30, ha="right")
        plt.tight_layout(); plt.savefig(ARTIFACT_DIR / "comparison.png", dpi=130); plt.close()
    except Exception:                                      # noqa: BLE001
        pass


# ============================ PREDICTION + XAI ============================
PLAIN = {
    "rain_sum_3h": "rain in the last 3 hours", "rain_sum_6h": "rain in the last 6 hours",
    "rain_sum_12h": "rain in the last 12 hours", "rain_sum_24h": "rain in the last 24 hours",
    "rain_sum_48h": "rain in the last 2 days", "rain_sum_72h": "rain in the last 3 days",
    "rain_sum_168h": "rain in the last 7 days", "rain_max_6h": "heaviest hour of rain recently",
    "rain_hours_24h": "hours of rain in the last day", "precipitation": "rain this hour",
    "soil_moisture_0_to_7cm": "soil wetness", "soil_mean_24h": "average soil wetness today",
    "soil_change_24h": "rise in soil wetness over a day",
    "relative_humidity_2m": "air humidity", "humidity_mean_6h": "recent air humidity",
    "surface_pressure": "air pressure", "pressure_change_3h": "falling/rising pressure (3 h)",
    "pressure_change_24h": "pressure change over a day", "temperature_2m": "temperature",
    "temp_mean_24h": "average temperature today", "wind_speed_10m": "wind speed",
    "elevation_m": "height of the land above sea level", "slope_degrees": "slope of the land",
    "relief_m": "height difference around the place",
    "elevation_above_min_m": "how much higher this spot is than its surroundings' lowest point",
    "hour_sin": "time of day", "hour_cos": "time of day",
    "doy_sin": "time of year (season)", "doy_cos": "time of year (season)",
}
UNITS = {"rain": "mm", "soil": "", "humidity": "%", "pressure": "hPa", "temp": "°C",
         "wind": "km/h", "elevation": "m", "relief": "m", "slope": "°"}


def _unit(f):
    for k, u in UNITS.items():
        if f.startswith(k) or k in f:
            return u
    return ""


def risk_band(p, thr):
    if p >= thr + 0.5 * (1 - thr): return "VERY HIGH"
    if p >= thr + 0.2 * (1 - thr): return "HIGH"
    if p >= thr:                   return "MODERATE"
    if p >= 0.5 * thr:             return "LOW"
    return "VERY LOW"


class FloodPredictor:
    """Loads the trained ensemble and predicts for a place / GPS point."""

    def __init__(self, artifact_dir: str | Path = ARTIFACT_DIR):
        d = Path(artifact_dir)
        if not (d / "config.json").exists():
            raise FileNotFoundError(f"No trained model in {d}. Run:  python {Path(__file__).name} --train")
        self.cfg = json.load(open(d / "config.json"))
        self.scaler = joblib.load(d / "scaler.joblib")
        self.names = self.cfg["model_names"]
        self.models = {n: tf.keras.models.load_model(str(d / f"{n}.keras")) for n in self.names}
        self.features = self.cfg["features"]
        self.thr = self.cfg["threshold"]

    # ---- core scoring ----
    def _score(self, X):                       # X: (n, W, F) scaled numpy
        P = np.stack([self.models[n].predict(X, verbose=0).ravel() for n in self.names], 1)
        return apply_ensemble(self.cfg["ensemble"], P)

    def _ens_tensor(self, x):
        ps = [self.models[n](x, training=False)[:, 0] for n in self.names]
        sp = self.cfg["ensemble"]
        if sp["type"] == "weighted":
            return tf.add_n([w * p for w, p in zip(sp["w"], ps)])
        z = sp["intercept"]
        for c, p in zip(sp["coef"], ps):
            p = tf.clip_by_value(p, 1e-6, 1 - 1e-6)
            z = z + c * tf.math.log(p / (1 - p))
        return tf.sigmoid(z)

    # ---- explainable AI: Integrated Gradients on the final ensemble ----
    def _integrated_gradients(self, X, steps=40):
        x = tf.convert_to_tensor(X, tf.float32)               # (1, W, F)
        base = tf.zeros_like(x)                               # 0 = typical (training mean)
        a = tf.reshape(tf.linspace(0.0, 1.0, steps + 1), (-1, 1, 1))
        inter = base + a * (x - base)
        with tf.GradientTape() as tape:
            tape.watch(inter)
            out = self._ens_tensor(inter)
        g = tape.gradient(out, inter)
        g = (g[:-1] + g[1:]) / 2.0
        return ((x - base)[0] * tf.reduce_mean(g, axis=0)).numpy()   # (W, F)

    def _explain(self, X, last_row, top=5):
        attr = self._integrated_gradients(X).sum(axis=0)
        # merge duplicate pairs (sin/cos) into one driver
        order = np.argsort(-np.abs(attr))
        drivers, seen = [], set()
        for i in order:
            f = self.features[i]
            label = PLAIN.get(f, f.replace("_", " "))
            if label in seen:
                continue
            seen.add(label)
            v = last_row[f]
            drivers.append({"feature": f, "label": label,
                            "value": round(float(v), 2), "unit": _unit(f),
                            "effect": "raises risk" if attr[i] > 0 else "lowers risk",
                            "strength": round(float(abs(attr[i])), 4)})
            if len(drivers) == top:
                break
        up = [d for d in drivers if d["effect"] == "raises risk"]
        dn = [d for d in drivers if d["effect"] == "lowers risk"]
        fmt = lambda d: f'{d["label"]} ({d["value"]}{(" " + d["unit"]) if d["unit"] else ""})'
        text = ""
        if up:
            text += "Flood risk is pushed UP by: " + "; ".join(fmt(d) for d in up) + ". "
        if dn:
            text += "It is held DOWN by: " + "; ".join(fmt(d) for d in dn) + "."
        return {"summary": text.strip() or "No single factor dominates.", "drivers": drivers}

    # ---- helpers ----
    def _features_for(self, lat, lon, on_date):
        w = fetch_weather(lat, lon, on_date)
        d = engineer(w, fetch_static(lat, lon))
        return d

    def _locate(self, place, lat, lon):
        if place:
            return geocode(place)
        if lat is None or lon is None:
            raise ValueError("Give a place name or lat + lon.")
        return float(lat), float(lon), f"{float(lat):.3f}, {float(lon):.3f}"

    @staticmethod
    def _now():
        return pd.Timestamp.now(tz="Asia/Kolkata").tz_localize(None).floor("h")

    def _window_ending(self, d, end_time):
        hist = d[d["time"] <= end_time]
        if len(hist) < WINDOW:
            raise ValueError("Not enough recent weather data for this place/date.")
        win = hist.tail(WINDOW)
        X = self.scaler.transform(win[self.features]).astype("float32")[None]
        return X, win

    # ---- public API ----
    def predict(self, place=None, lat=None, lon=None, on_date=None, explain=True):
        lat, lon, label = self._locate(place, lat, lon)
        d = self._features_for(lat, lon, on_date)
        end = (pd.Timestamp(on_date).normalize() + pd.Timedelta(hours=23)) if on_date else self._now()
        X, win = self._window_ending(d, end)
        p = float(self._score(X)[0])
        last = win.iloc[-1]
        fut = d[(d["time"] > end) & (d["time"] <= end + pd.Timedelta(hours=24))]
        out = {"place": label, "latitude": round(lat, 4), "longitude": round(lon, 4),
               "as_of": str(win["time"].iloc[-1]),
               "flood_risk_score": round(p, 3), "risk_level": risk_band(p, self.thr),
               "flood_expected_next_24h": bool(p >= self.thr),
               "alert_threshold": round(self.thr, 3),
               "conditions_now": {"rain_last_24h_mm": round(float(last["rain_sum_24h"]), 1),
                                  "rain_last_72h_mm": round(float(last["rain_sum_72h"]), 1),
                                  "soil_wetness": round(float(last["soil_moisture_0_to_7cm"]), 3),
                                  "elevation_m": round(float(last["elevation_m"]), 1)},
               "forecast_rain_next_24h_mm": round(float(fut["precipitation"].sum()), 1) if len(fut) else None,
               "model": {"ensemble": self.cfg["ensemble_name"], "members": self.names,
                         "trained_on": self.cfg["data_source"], "label_type": self.cfg["label_type"]},
               "disclaimer": "Early-warning risk score, not an official forecast. Follow IMD/CWC/local alerts."}
        if explain:
            out["explanation"] = self._explain(X, last)
        return out

    def hourly_strip(self, place=None, lat=None, lon=None, explain=True):
        """Risk for each of the next 24 hours (use for the clickable hourly strip)."""
        lat, lon, label = self._locate(place, lat, lon)
        d = self._features_for(lat, lon, None)
        now = self._now()
        rows, Xs, wins = [], [], []
        for h in range(1, 25):
            t = now + pd.Timedelta(hours=h)
            try:
                X, win = self._window_ending(d, t)
            except ValueError:
                continue
            Xs.append(X[0]); wins.append((t, win))
        if not Xs:
            return []
        P = self._score(np.stack(Xs))
        for (t, win), X, p in zip(wins, Xs, P):
            row = {"hour": str(t), "risk_score": round(float(p), 3),
                   "risk_level": risk_band(float(p), self.thr),
                   "rain_mm": round(float(win.iloc[-1]["precipitation"]), 2)}
            if explain:
                row["reason"] = self._explain(X[None], win.iloc[-1], top=3)["summary"]
            rows.append(row)
        return rows


# ================================== CLI ===================================
def main():
    ap = argparse.ArgumentParser(description="FloodXplain model")
    ap.add_argument("--train", action="store_true", help="download data, train, test, save")
    ap.add_argument("--synthetic", action="store_true", help="force synthetic data (offline test)")
    ap.add_argument("--epochs", type=int, default=EPOCHS)
    ap.add_argument("--predict", type=str, help="place name in India")
    ap.add_argument("--lat", type=float); ap.add_argument("--lon", type=float)
    ap.add_argument("--date", type=str, help="past date YYYY-MM-DD (default: now)")
    ap.add_argument("--strip", action="store_true", help="also print the 24-hour strip")
    a, unknown = ap.parse_known_args()

    if a.train:
        train("synthetic" if a.synthetic else DATA_MODE, a.epochs)
    if a.predict or a.lat is not None:
        fp = FloodPredictor()
        print(json.dumps(fp.predict(place=a.predict, lat=a.lat, lon=a.lon, on_date=a.date),
                         indent=2, default=str))
        if a.strip:
            print(json.dumps(fp.hourly_strip(place=a.predict, lat=a.lat, lon=a.lon),
                             indent=2, default=str))
    if not (a.train or a.predict or a.lat is not None):
        ap.print_help()


if __name__ == "__main__":
    main()


usage: colab_kernel_launcher.py [-h] [--train] [--synthetic] [--epochs EPOCHS]
                                [--predict PREDICT] [--lat LAT] [--lon LON]
                                [--date DATE] [--strip]

FloodXplain model

options:
  -h, --help         show this help message and exit
  --train            download data, train, test, save
  --synthetic        force synthetic data (offline test)
  --epochs EPOCHS
  --predict PREDICT  place name in India
  --lat LAT
  --lon LON
  --date DATE        past date YYYY-MM-DD (default: now)
  --strip            also print the 24-hour strip


In [ ]:
!pwd
!ls -lah /content

/content
total 16K
drwxr-xr-x 1 root root 4.0K Oct 26 22:18 .
drwxr-xr-x 1 root root 4.0K Nov  4 05:10 ..
drwxr-xr-x 4 root root 4.0K Oct 26 22:18 .config
drwxr-xr-x 1 root root 4.0K Oct 26 22:18 sample_data


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving floodxplain_model.py to floodxplain_model.py


In [ ]:
!ls -lh /content/floodxplain_model.py

-rw-r--r-- 1 root root 37K Nov  4 05:13 /content/floodxplain_model.py


In [ ]:
!python -m py_compile /content/floodxplain_model.py

In [ ]:
!python /content/floodxplain_model.py --train --epochs 20

I0000 00:00:1793769336.364965    7210 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793769341.229825    7210 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
== 1/5 Loading data ==
  downloading Coimbatore ...
  downloading Saravanampatti ...
  downloading Chennai ...
  downloading Alappuzha ...
  downloading Wayanad ...
  downloading Bengaluru ...
  downloading Hyderabad ...
  downloading Mumbai ...
  downloading Surat ...
  downloading Kolkata ...
  downloading Cuttack ...
  downloading Guwahati ...
  downloading Dibrugarh ...
  downloading Patna ...
  downloading Darbhanga ...
  downloading Delhi ...
  downloading Dehradun ...
  downloading Srinagar ...
  downloading Jaipur ...
  downloading Jaisalmer ...
== 2/5 Feature engineering, labels, split ==

Data source : REAL (Open-Meteo ERA5)
Label type  : PROXY (IMD rainfall rule)
Locations   : 20   rows: 1,368,500   features: 29
  train:   957,020 

In [4]:
!pwd
!ls -lah /content
!find /content -maxdepth 3 -type d -name "*artifact*"

/content
total 16K
drwxr-xr-x 1 root root 4.0K Oct 26 22:18 .
drwxr-xr-x 1 root root 4.0K Nov  4 06:40 ..
drwxr-xr-x 4 root root 4.0K Oct 26 22:18 .config
drwxr-xr-x 1 root root 4.0K Oct 26 22:18 sample_data


In [5]:
!ls -lah /content/floodxplain_artifacts

ls: cannot access '/content/floodxplain_artifacts': No such file or directory


In [6]:
!find /content -maxdepth 5 -type f \( \
-name "*.keras" -o \
-name "*.h5" -o \
-name "*.pkl" -o \
-name "*.joblib" -o \
-name "*.json" \
\) -print

/content/.config/.last_update_check.json
/content/sample_data/anscombe.json


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
!find /content/drive/MyDrive -maxdepth 5 -type d -iname "*floodxplain*"

In [10]:
!find /content/drive/MyDrive -maxdepth 6 \
\( -iname "*floodxplain*" -o -iname "*.keras" -o -iname "*.h5" \) \
-print

/content/drive/MyDrive/Colab Notebooks/FloodXplain_Phase5A_Data_Acquisition (1).ipynb
/content/drive/MyDrive/Colab Notebooks/FloodXplain_Phase5A_Data_Acquisition.ipynb
/content/drive/MyDrive/DL_Experiment_24BAD022/mnist_deep_learning_model.keras
/content/drive/MyDrive/TransferLearning_24BAD022.keras


In [11]:
import os

DRIVE_ARTIFACTS = "/content/drive/MyDrive/FloodXplain/floodxplain_artifacts"

os.makedirs(DRIVE_ARTIFACTS, exist_ok=True)

print("Artifact folder:", DRIVE_ARTIFACTS)

Artifact folder: /content/drive/MyDrive/FloodXplain/floodxplain_artifacts


In [12]:
!grep -n -E "floodxplain_artifacts|ARTIFACT|OUTPUT_DIR|save_dir" /content/floodxplain_model.py

grep: /content/floodxplain_model.py: No such file or directory
